In [ ]:



# Cell 2: Import all necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_nlp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

print("✅ Libraries imported successfully.")

# Cell 3: Load the data
try:
    train_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/train.csv")
    test_df = pd.read_csv("/kaggle/input/map-charting-student-math-misunderstandings/test.csv")
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("🛑 Data files not found. Make sure the competition data is added to your notebook.")

In [ ]:
# Cell 4: Prepare Text and Labels

# 1. Combine text fields into a single input string
train_df['input_text'] = train_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)
test_df['input_text'] = test_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)

# 2. Prepare labels for multi-label classification
# Ensure columns are strings to prevent errors
train_df['Category'] = train_df['Category'].astype(str)
train_df['Misconception'] = train_df['Misconception'].astype(str)
train_df['full_label'] = train_df['Category'] + ':' + train_df['Misconception']

# Use MultiLabelBinarizer to create multi-hot encoded vectors
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(train_df['full_label'].apply(lambda x: [x]))

print("✅ Text and labels prepared.")

In [ ]:
# Cell 5: Create a Train/Validation Split
X_train, X_val, y_train, y_val = train_test_split(
    train_df['input_text'], 
    y_train, 
    test_size=0.2, 
    random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

In [ ]:
MODEL_PATH = "/kaggle/input/deberta_v3/keras/deberta_v3_base_en/3"

# Calculate the number of unique labels
num_labels = len(mlb.classes_)

# Load the entire classifier in one go. 
# It automatically includes the preprocessor and a classification head.
classifier = keras_nlp.models.DebertaV3Classifier.from_preset(
    "deberta_v3_base_en",
    num_classes=num_labels,
)

# Compile the model. By using "binary_crossentropy" as the loss, Keras
# will automatically use a 'sigmoid' activation, which is correct for multi-label tasks.
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="binary_crossentropy",
    metrics=["binary_accuracy"]
)

classifier.summary()
print("✅ DebertaV3Classifier loaded and compiled successfully.")


In [ ]:
# Cell 4: Manually Create TensorFlow Datasets
BATCH_SIZE = 8

# Create the training dataset from our pandas/numpy objects
tf_train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
# Shuffle, batch, and prefetch for optimal performance
tf_train_dataset = tf_train_dataset.shuffle(buffer_size=len(X_train)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create the validation dataset
tf_val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))
tf_val_dataset = tf_val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ Data converted to the tf.data.Dataset format.")

In [ ]:
print("⏳ Starting model training...")

# The .fit() command starts the training process.
# - epochs: How many times the model will see the entire training dataset.
# - batch_size: How many samples the model works on at once.
# Cell 5: Train the Model
history = classifier.fit(
    tf_train_dataset,
    validation_data=tf_val_dataset,
    epochs=3
)

print("✅ Model training complete.")

print("✅ Model training complete.")

In [ ]:
# Cell: Prediction and Submission

print("⏳ Generating predictions on the test set...")

# 1. Prepare the test data's input text column
test_df['input_text'] = test_df.apply(lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", axis=1)

# 2. Get the predicted probabilities from the trained classifier
# The classifier automatically preprocesses the raw text.
test_probabilities = classifier.predict(test_df['input_text'])

# 3. Find the indices of the top 3 probabilities for each row
top_3_indices = np.argsort(test_probabilities, axis=1)[:, ::-1][:, :3]

# 4. Use the MultiLabelBinarizer (mlb) to convert indices back to label names
top_3_labels = mlb.classes_[top_3_indices]

# 5. Join the 3 labels with a space to match the submission format
predictions_str = [" ".join(labels) for labels in top_3_labels]

# 6. Create the final submission DataFrame
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'], 
    'Category:Misconception': predictions_str
})

# 7. Save the DataFrame to a csv file
submission_df.to_csv('submission.csv', index=False)

print("✅ submission.csv file created successfully!")
print("Here's a preview:")
print(submission_df.head())